# Snap-AI: train a face embedder with ArcFace on Colab
Runtime > Change runtime type > GPU (A100/L4 if offered). Run the cells top to bottom.
Everything important is saved to Google Drive (`MyDrive/snap_ai`) so a disconnect does not lose work: just re-run the
setup cells and the `--resume` training cell.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print(torch.__version__, torch.cuda.is_available())

## 1. Drive + code
Put `snap_ai_code.zip` (from the project folder) in `MyDrive/snap_ai/` first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/snap_ai'
os.makedirs(DRIVE + '/runs', exist_ok=True)
!rm -rf /content/code && mkdir -p /content/code && cd /content/code && unzip -q -o {DRIVE}/snap_ai_code.zip && ls
%cd /content/code
!pip -q install opencv-python-headless pillow

## 2. Data (downloaded straight into the Colab VM, not your PC)
Paste the dataset link into `DATA_URL`. A Google Drive link works with `gdown`; any direct https link works with `wget`.
Expected pack contents: `train.rec`, `train.idx`, `lfw.bin`, `cfp_fp.bin`, `agedb_30.bin` (InsightFace `faces_webface_112x112`).
Alternatively copy the pack from your own Drive into `/content/pack` instead of downloading.

In [ ]:
# Use ONE of the two options. Option A (recommended): your own Drive copy of the pack file.
DRIVE_FILE = ''   # e.g. '/content/drive/MyDrive/CASIA-Webface.zip'  (right-click the file in the Colab Files panel > Copy path)
DATA_URL = ''     # Option B: Google Drive / direct link, only if you did not make a copy
import os
os.makedirs('/content/pack', exist_ok=True)
if not os.path.exists('/content/pack/train.rec') and not any('train.rec' in f for _, _, fs in os.walk('/content/pack') for f in fs):
    if DRIVE_FILE:
        !cp "{DRIVE_FILE}" /content/pack/
    elif DATA_URL:
        if 'drive.google.com' in DATA_URL:
            !pip -q install gdown
            !cd /content/pack && gdown --fuzzy "{DATA_URL}"
        else:
            !cd /content/pack && wget -q --show-progress "{DATA_URL}"
    else:
        raise SystemExit('Set DRIVE_FILE or DATA_URL first')
    !cd /content/pack && for f in *.zip; do [ -f "$f" ] && unzip -q -o "$f"; done; for f in *.tar *.tar.gz; do [ -f "$f" ] && tar xf "$f"; done; true
!find /content/pack -name "*.rec" -o -name "*.bin" | head

In [ ]:
# locate the pack folder (it may be nested one level) and convert .rec -> image folders on the FAST local disk
import glob
rec = glob.glob('/content/pack/**/train.rec', recursive=True)[0]
PACK = os.path.dirname(rec); print('pack:', PACK)
if not os.path.exists('/content/casia_112'):
    !python tools/rec_to_folder.py --rec {rec} --out /content/casia_112
!ls /content/casia_112 | head -3; ls /content/casia_112 | wc -l; ls /content/casia_112/$(ls /content/casia_112 | head -1) | head -3

**Spot-check** a few converted images (they should be aligned 112x112 face crops, same person within a folder):

In [ ]:
import glob, random
from PIL import Image
import matplotlib.pyplot as plt
ids = sorted(os.listdir('/content/casia_112'))[:4]
fig, ax = plt.subplots(len(ids), 6, figsize=(9, 1.6*len(ids)))
for r, i in enumerate(ids):
    fs = sorted(glob.glob(f'/content/casia_112/{i}/*'))[:6]
    for c in range(6):
        ax[r][c].axis('off')
        if c < len(fs): ax[r][c].imshow(Image.open(fs[c]))
plt.show()

## 3. Quick trial (about 10-20 min): does it learn on real faces?
LFW accuracy should climb well above 50% within a few epochs.

In [ ]:
!python -m train.train --data /content/casia_112 --val-bins {PACK}/lfw.bin \
    --arch r18 --max-ids 2000 --epochs 8 --batch 128 --workers 4 --out {DRIVE}/runs/trial_r18

## 4. Full training run (resumable)
Re-run this cell after a disconnect; `--resume` continues from `last.pt` in Drive.

In [ ]:
!python -m train.train --data /content/casia_112 \
    --val-bins {PACK}/lfw.bin {PACK}/cfp_fp.bin {PACK}/agedb_30.bin \
    --arch r34 --epochs 25 --batch 256 --workers 4 --resume --out {DRIVE}/runs/r34_casia

## 5. Ablations (shortened schedule so the whole grid fits)

In [ ]:
!python -m train.ablate run --data /content/casia_112 \
    --val-bins {PACK}/lfw.bin {PACK}/cfp_fp.bin \
    --epochs 8 --root {DRIVE}/runs/ablation --extra --max-ids 2000 --batch 128 --workers 4

In [ ]:
!python -m train.ablate summarize --root {DRIVE}/runs/ablation

## 5b. Fix for the hard-case collapse: fine-tune with quality augmentation
Starts from the finished `r34_casia/last.pt`; 50% of images get random low-res / blur / JPEG (milder than the hardest test cases). About 2-3 h on a T4; resumable.

In [ ]:
!python -m train.train --data /content/casia_112 \
    --val-bins {PACK}/lfw.bin {PACK}/cfp_fp.bin {PACK}/agedb_30.bin \
    --arch r34 --epochs 5 --batch 256 --workers 4 --lr 0.02 --warmup-epochs 0.3 --margin-warmup-epochs 0 \
    --degrade-p 0.5 --init {DRIVE}/runs/r34_casia/last.pt --resume --out {DRIVE}/runs/r34_casia_aug

## 6. Hard cases: ours vs. pretrained
The pretrained baseline needs `w600k_r50.onnx` from InsightFace `buffalo_l` (`pip install onnxruntime-gpu insightface`, then it downloads to `~/.insightface/models/buffalo_l/`).

In [ ]:
!pip -q install onnxruntime-gpu insightface
from insightface.app import FaceAnalysis
FaceAnalysis(name='buffalo_l', allowed_modules=['detection'])   # triggers the model download
ONNX = os.path.expanduser('~/.insightface/models/buffalo_l/w600k_r50.onnx'); print(os.path.exists(ONNX))

In [ ]:
!ls {PACK}/*.bin
!python -m train.hard_cases --ckpt {DRIVE}/runs/r34_casia/best.pt \
    --degrade-bin {PACK}/lfw.bin --extra-bins {PACK}/cfp_fp.bin {PACK}/agedb_30.bin \
    --onnx {ONNX} --out {DRIVE}/runs/hard_cases.md
!cat {DRIVE}/runs/hard_cases.md

### Compare before vs after augmentation (and vs pretrained)
Uses `last.pt` of the fine-tuned model (final epoch), not `best.pt`, so the checkpoint is not chosen by a clean-data metric. `--dump-examples` saves pictures of each degradation for a sanity check.

In [ ]:
!python -m train.hard_cases \
    --ckpt base={DRIVE}/runs/r34_casia/best.pt aug={DRIVE}/runs/r34_casia_aug/last.pt \
    --degrade-bin {PACK}/lfw.bin --extra-bins {PACK}/cfp_fp.bin {PACK}/agedb_30.bin \
    --onnx {ONNX} --out {DRIVE}/runs/hard_cases_aug.md --dump-examples {DRIVE}/runs/degrade_examples
!cat {DRIVE}/runs/hard_cases_aug.md